# Source-disjoint partition, three seeds, explanation fidelity

Runs `tools/source_disjoint/PROTOCOL.md` end to end on a GPU runtime. Drive holds `polyglotfake/processed/{train,val,test}` (the released clip-level partition, 3,452 clips). Outputs go to Drive `polyglotfake/source_disjoint/` so the run survives a disconnect; every stage resumes.

Runtime on a T4: roughly 2-3 h per seed for the five agents, plus about 20 min of scoring per seed; the fine-tune stage adds about 1 h per seed. Run the seeds in separate sessions if the runtime is recycled.

In [ ]:
!nvidia-smi -L
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
%cd /content
!rm -rf repo && git clone -q -b revision/round-3 https://github.com/saoirsebarry/multiagent-deepfake-detection.git repo
%cd /content/repo
!pip -q install speechbrain timm librosa opencv-python-headless facenet-pytorch
!git log --oneline -1

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
PROCESSED = '/content/drive/MyDrive/polyglotfake/processed'
OUT = '/content/drive/MyDrive/polyglotfake/source_disjoint'
os.makedirs(OUT, exist_ok=True)
for s in ('train', 'val', 'test'):
    d = os.path.join(PROCESSED, s); print(f'{s:5s} {len(os.listdir(d)) if os.path.isdir(d) else 0:5d} clips')

## Stage the released clips on local disk
All three released partitions are needed: the new partition draws from every clip.

In [ ]:
!python -u tools/robust_finetune/stage_data.py --src "$PROCESSED" --dst /content/pgf --splits train val test
!du -sh /content/pgf/*

## Identity groups and the new partition
`cluster_identities.py` links source videos whose authentic clips show the same face (VGGFace2 InceptionResnetV1, cosine distance below 0.4). `make_split.py` then assigns whole identity groups to one partition. The manifest hash is recorded before any training.

In [ ]:
!python -u tools/source_disjoint/cluster_identities.py --processed /content/pgf --out "$OUT/identity_groups.json"
!python -u tools/source_disjoint/make_split.py --processed /content/pgf --out_dir /content/pgf_sd --seed 42 --groups "$OUT/identity_groups.json" --manifest "$OUT/split_manifest.json" --audit

## Train, fine-tune and score, one seed per cell
Each cell is resumable. `--finetune` runs the released robustness fine-tunes warm-started from this run's own checkpoints; `--score` writes `scores_val.csv` and `scores_test.csv`.

In [ ]:
SEED = 42
!python -u tools/source_disjoint/train_all.py --data_dir /content/pgf_sd --out_dir "$OUT/seed$SEED" --seed $SEED --finetune --score 2>&1 | tail -n 30

In [ ]:
SEED = 43
!python -u tools/source_disjoint/train_all.py --data_dir /content/pgf_sd --out_dir "$OUT/seed$SEED" --seed $SEED --finetune --score 2>&1 | tail -n 30

In [ ]:
SEED = 44
!python -u tools/source_disjoint/train_all.py --data_dir /content/pgf_sd --out_dir "$OUT/seed$SEED" --seed $SEED --finetune --score 2>&1 | tail -n 30

## Read-out
Threshold set on each run's validation scores, one test read per run, source-clustered intervals, McNemar tests, mean and standard deviation across seeds.

In [ ]:
!python -u tools/source_disjoint/analyse.py --runs "$OUT/seed42" "$OUT/seed43" "$OUT/seed44" --out "$OUT/readout.json"

## Explanation fidelity and stability on the released system
Runs on the released test partition with the released checkpoints (the numbers the paper's Section 4.7 describes).

In [ ]:
!python -u tools/xai_fidelity/run_fidelity.py --data_dir /content/pgf --split test --ckpt_dir checkpoints --n 60 --out "$OUT/xai_fidelity_released.json" 2>&1 | tail -n 40

## Collect
The JSON read-outs are on Drive; download them from the Drive web UI if `files.download` stalls.

In [ ]:
from google.colab import files
for f in ['readout.json', 'split_manifest.json', 'identity_groups.json', 'xai_fidelity_released.json']:
    p = os.path.join(OUT, f)
    if os.path.exists(p):
        try: files.download(p)
        except Exception as e: print('skip', f, e)